# 05b — Wrapper Methods

## The One-Sentence Version
Instead of scoring features individually, test combinations to find the best subset.

## What You'll Build Intuition For
- Forward selection: start empty, add the best feature one at a time
- Backward elimination: start full, remove the worst feature one at a time
- Recursive Feature Elimination (RFE): use model importance to prune
- The computational cost of searching through feature combinations

## Prerequisites
- [05a — Filter Methods](05a_filter_methods.ipynb) — individual feature scoring

## The Story

Filter methods score each feature on its own. But what if feature A is useless alone, yet powerful when combined with feature B? Filters would throw A away.

Wrapper methods take a different approach: they actually *train a model* on different feature subsets and measure which subset performs best. It's like auditioning musicians — instead of testing each one alone, you try different combinations to find the best ensemble.

The downside? It's expensive. Every combination requires training a model. But when you can afford it, wrappers find better subsets than filters.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE, RFECV

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_patient_data

apply_style()
rng = np.random.default_rng(42)

In [ ]:
# Dataset: 5 signal + 15 noise features
X, feature_names, signal_mask = make_patient_data(
    n=400, d_signal=5, d_noise=15, seed=42
)
y = (X[:, 0] > np.median(X[:, 0])).astype(int)  # binary target from age

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features: {len(feature_names)} ({signal_mask.sum()} signal, {(~signal_mask).sum()} noise)")
print(f"Signal features: {[n for n, s in zip(feature_names, signal_mask) if s]}")

## Forward Selection

Start with no features. At each step, try adding each remaining feature and keep the one that improves cross-validation accuracy the most. Stop when adding more doesn't help.

In [ ]:
def forward_selection(X, y, feature_names, max_features=None):
    """Greedy forward feature selection with cross-validation."""
    n_features = X.shape[1]
    if max_features is None:
        max_features = n_features

    selected = []
    remaining = list(range(n_features))
    scores = []
    selected_at_step = []

    model = LogisticRegression(max_iter=1000, random_state=42)

    for step in range(min(max_features, n_features)):
        best_score = -1
        best_feat = None

        for feat in remaining:
            trial = selected + [feat]
            score = cross_val_score(model, X[:, trial], y, cv=5).mean()
            if score > best_score:
                best_score = score
                best_feat = feat

        selected.append(best_feat)
        remaining.remove(best_feat)
        scores.append(best_score)
        selected_at_step.append(feature_names[best_feat])

    return selected, scores, selected_at_step

fwd_idx, fwd_scores, fwd_names = forward_selection(X_scaled, y, feature_names)

fig, ax = plt.subplots(figsize=(12, 5))
bar_colours = [COLOURS["signal"] if signal_mask[i] else COLOURS["noise"] for i in fwd_idx]
ax.plot(range(1, len(fwd_scores) + 1), fwd_scores, "o-", color=COLOURS["signal"],
        linewidth=2, markersize=6)

# Mark signal vs noise features
for i, (score, idx) in enumerate(zip(fwd_scores, fwd_idx)):
    colour = COLOURS["signal"] if signal_mask[idx] else COLOURS["noise"]
    ax.scatter(i + 1, score, color=colour, s=80, zorder=5)

ax.set_xlabel("Number of Features")
ax.set_ylabel("CV Accuracy")
ax.set_title("Forward Selection: signal features (blue) are added first")
ax.set_xticks(range(1, len(fwd_scores) + 1))
ax.set_xticklabels([f"{i+1}\n{n}" for i, n in enumerate(fwd_names)],
                    fontsize=7, rotation=45, ha="right")

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=COLOURS["signal"], label="Signal feature"),
    Patch(facecolor=COLOURS["noise"], label="Noise feature"),
])

plt.tight_layout()
plt.savefig("visuals/05b_forward_selection.png", dpi=150, bbox_inches="tight")
plt.show()

print("Order features were added:")
for i, (name, score) in enumerate(zip(fwd_names, fwd_scores), 1):
    tag = "SIGNAL" if signal_mask[fwd_idx[i-1]] else "noise"
    print(f"  {i}. {name} ({tag}) → accuracy {score:.3f}")

## Backward Elimination

The opposite approach: start with all features and remove the least useful one at each step. This can find different subsets than forward selection — neither is guaranteed to find the global optimum.

In [ ]:
def backward_elimination(X, y, feature_names, min_features=1):
    """Greedy backward feature elimination with cross-validation."""
    remaining = list(range(X.shape[1]))
    scores = []
    removed_at_step = []

    model = LogisticRegression(max_iter=1000, random_state=42)

    # Initial score with all features
    score_all = cross_val_score(model, X[:, remaining], y, cv=5).mean()
    scores.append(score_all)
    removed_at_step.append("(all)")

    while len(remaining) > min_features:
        worst_score = -1
        worst_feat = None

        for feat in remaining:
            trial = [f for f in remaining if f != feat]
            score = cross_val_score(model, X[:, trial], y, cv=5).mean()
            if score > worst_score:
                worst_score = score
                worst_feat = feat

        removed_at_step.append(feature_names[worst_feat])
        remaining.remove(worst_feat)
        scores.append(worst_score)

    return remaining, scores, removed_at_step

bwd_idx, bwd_scores, bwd_removed = backward_elimination(X_scaled, y, feature_names)

fig, ax = plt.subplots(figsize=(12, 5))
n_feats = list(range(len(feature_names), 0, -1))
ax.plot(n_feats, bwd_scores, "o-", color=COLOURS["accent"], linewidth=2, markersize=6)
ax.set_xlabel("Number of Features Remaining")
ax.set_ylabel("CV Accuracy")
ax.set_title("Backward Elimination: accuracy holds as noise features are removed")
ax.invert_xaxis()

plt.tight_layout()
plt.savefig("visuals/05b_backward_elimination.png", dpi=150, bbox_inches="tight")
plt.show()

print("Final surviving features:")
for i in bwd_idx:
    tag = "SIGNAL" if signal_mask[i] else "noise"
    print(f"  {feature_names[i]} ({tag})")

# Compare forward and backward selections
fwd_set = set(fwd_idx[:5])
bwd_set = set(bwd_idx) if len(bwd_idx) <= 5 else set(bwd_idx[:5])
print(f"\nForward (top 5):  {sorted([feature_names[i] for i in fwd_set])}")
print(f"Backward (final): {sorted([feature_names[i] for i in bwd_set])}")
print(f"Agreement: {len(fwd_set & bwd_set)}/{min(len(fwd_set), len(bwd_set))} features overlap")

## Recursive Feature Elimination (RFE)

RFE is a smarter version of backward elimination: instead of trying every removal, it trains a model once, looks at the feature importances (coefficient magnitudes), and removes the weakest feature. Repeat until the desired number remains.

`RFECV` goes further: it uses cross-validation to find the *optimal* number of features.

In [ ]:
# RFE with cross-validation to find optimal number of features
model = LogisticRegression(max_iter=1000, random_state=42)
rfecv = RFECV(estimator=model, step=1, cv=5, scoring="accuracy", min_features_to_select=1)
rfecv.fit(X_scaled, y)

fig, ax = plt.subplots(figsize=(10, 5))
n_features_range = range(1, len(rfecv.cv_results_["mean_test_score"]) + 1)
ax.plot(n_features_range, rfecv.cv_results_["mean_test_score"], "o-",
        color=COLOURS["signal"], linewidth=2, markersize=5)
ax.fill_between(n_features_range,
                rfecv.cv_results_["mean_test_score"] - rfecv.cv_results_["std_test_score"],
                rfecv.cv_results_["mean_test_score"] + rfecv.cv_results_["std_test_score"],
                alpha=0.2, color=COLOURS["signal"])
ax.axvline(x=rfecv.n_features_, color=COLOURS["accent"], linestyle="--",
           label=f"Optimal: {rfecv.n_features_} features")
ax.set_xlabel("Number of Features")
ax.set_ylabel("CV Accuracy")
ax.set_title("RFECV: finding the optimal number of features")
ax.legend()
plt.tight_layout()
plt.savefig("visuals/05b_rfecv.png", dpi=150, bbox_inches="tight")
plt.show()

# Which features survived?
rfe_selected = [feature_names[i] for i in range(len(feature_names)) if rfecv.support_[i]]
rfe_signal = [n for n, s in zip(feature_names, signal_mask) if s and rfecv.support_[list(feature_names).index(n) if isinstance(feature_names, list) else n]]

print(f"Optimal number of features: {rfecv.n_features_}")
print(f"Selected features: {rfe_selected}")
print(f"\nFeature rankings (1 = selected):")
for name, rank, is_signal in zip(feature_names, rfecv.ranking_, signal_mask):
    tag = "SIGNAL" if is_signal else "noise"
    marker = "✓" if rank == 1 else " "
    print(f"  [{marker}] rank {rank:>2}: {name} ({tag})")

## The Cost of Wrapper Methods

Wrapper methods are powerful but expensive. Let's see why.

In [ ]:
# Count model trainings required
d = len(feature_names)
cv_folds = 5

# Forward: step k evaluates (d - k) candidates × cv_folds
fwd_trainings = sum((d - k) * cv_folds for k in range(d))
# Backward: step k evaluates (d - k) candidates × cv_folds
bwd_trainings = sum((d - k) * cv_folds for k in range(d))
# RFE: d steps × cv_folds trainings each
rfe_trainings = d * cv_folds

fig, ax = plt.subplots(figsize=(10, 5))
methods = ["Forward\nSelection", "Backward\nElimination", "RFE\n(no CV)", "RFECV"]
trainings = [fwd_trainings, bwd_trainings, d, rfe_trainings]
colours_bar = [COLOURS["noise"], COLOURS["noise"], COLOURS["signal"], COLOURS["accent"]]

ax.bar(methods, trainings, color=colours_bar, alpha=0.8, edgecolor="white")
for i, t in enumerate(trainings):
    ax.text(i, t + max(trainings) * 0.02, f"{t:,}", ha="center", fontsize=11,
            fontweight="bold")
ax.set_ylabel("Model Trainings Required")
ax.set_title(f"Computational cost for d={d} features, {cv_folds}-fold CV")
plt.tight_layout()
plt.savefig("visuals/05b_cost.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"With {d} features and {cv_folds}-fold CV:")
print(f"  Forward selection: {fwd_trainings:,} model trainings")
print(f"  Backward elimination: {bwd_trainings:,} model trainings")
print(f"  RFE (no CV): {d} trainings")
print(f"  RFECV: {rfe_trainings} trainings")
print(f"\nWith 1000 features, forward/backward would need ~2.5 million trainings.")
print(f"Rule of thumb: use wrappers when d < 50, filters or embedded methods otherwise.")

> **Key insight:** Forward and backward selection are greedy — they find *locally* optimal subsets, not globally optimal ones. Forward may miss features that are only useful together. Backward may keep features that are only useful individually. Neither explores the full combinatorial space (which is 2^d possibilities).

<details>
<summary><b>The Maths Behind This</b></summary>

**The search space:** With $d$ features, there are $2^d$ possible subsets. Exhaustive search is NP-hard for large $d$.

**Forward selection** is a greedy algorithm:

$$S_0 = \emptyset, \quad S_{k+1} = S_k \cup \{\arg\max_{j \notin S_k} \text{score}(S_k \cup \{j\})\}$$

Cost: $O(d^2)$ model evaluations ($d + (d-1) + \ldots + 1$).

**RFE** avoids the nested evaluation by using the model's own importance scores:

1. Fit model on current features
2. Rank features by $|w_j|$ (coefficient magnitude)
3. Remove the feature with smallest $|w_j|$
4. Repeat

Cost: $O(d)$ model trainings. Much cheaper, but depends on the base model providing good importance estimates.

</details>

## Where This Matters

**Clinical prediction models:** When building a sepsis predictor, clinicians need to know *which specific labs to order*. A 5-feature model that uses creatinine, lactate, WBC, temperature, and heart rate is actionable. A 200-feature model is not. Wrapper methods find the smallest set that achieves target accuracy.

**Sensor placement:** In engineering, each sensor costs money. RFE can determine which sensors are redundant, reducing hardware costs while maintaining monitoring quality.

## Feynman Check

1. **Why might forward and backward selection give different results?**

2. **When is the computational cost of wrapper methods justified?**

---

Wrapper methods test feature combinations but are expensive. In **05c — Embedded Methods**, we'll see models that do feature selection *as a side effect of training* — the best of both worlds.